# 2 — Preprocessing

Raw text cannot be fed to a model. Three things have to happen first:

1. **Split sentences into words** — harder than it sounds for medical text.
2. **Split abstracts into sentences** — also harder than it sounds.
3. **Turn the corpus's character positions into one tag per word** — the Stage 2 labels.

All three live in `src/tokenizer.py` and `src/bio.py`. This notebook runs them on real
examples so you can see exactly what they do.

In [1]:
import sys, json, textwrap
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd
pd.set_option("display.max_colwidth", 90)
print("project root:", ROOT)

project root: E:\CSE\NLP Project


---

## 2.1 Why the project does not use an off-the-shelf tokenizer

A standard tokenizer is `re.findall(r'[A-Za-z0-9]+')` plus `.lower()`. On ordinary English
that is fine. On medical text it destroys precisely the words that carry the meaning.

In [2]:
from src.tokenizer import tokenize, naive_tokenize

hard = [
    "Treatment with 5-fluorouracil raised TNF-alpha levels.",
    "The dose was 20 mg/kg per day.",
    "The difference was significant (P<0.05).",
    "The patient had ALL and was given MTX.",
]

for sentence in hard:
    print(sentence)
    print("   naive :", naive_tokenize(sentence))
    print("   ours  :", tokenize(sentence)[0])
    print()

Treatment with 5-fluorouracil raised TNF-alpha levels.
   naive : ['treatment', 'with', '5', 'fluorouracil', 'raised', 'tnf', 'alpha', 'levels']
   ours  : ['treatment', 'with', '5-fluorouracil', 'raised', 'tnf-alpha', 'levels', '.']

The dose was 20 mg/kg per day.
   naive : ['the', 'dose', 'was', '20', 'mg', 'kg', 'per', 'day']
   ours  : ['the', 'dose', 'was', '20 mg/kg', 'per', 'day', '.']

The difference was significant (P<0.05).
   naive : ['the', 'difference', 'was', 'significant', 'p', '0', '05']
   ours  : ['the', 'difference', 'was', 'significant', '(', 'p<0.05', ')', '.']

The patient had ALL and was given MTX.
   naive : ['the', 'patient', 'had', 'all', 'and', 'was', 'given', 'mtx']
   ours  : ['the', 'patient', 'had', 'ALL', 'and', 'was', 'given', 'MTX', '.']



Read the four cases:

| Input | What the naive tokenizer does | Why it matters |
|---|---|---|
| `5-fluorouracil` | `5` + `fluorouracil` | the drug name no longer exists as a word |
| `TNF-alpha` | `tnf` + `alpha` | same, for a protein target |
| `20 mg/kg` | four meaningless fragments | dosage information is lost |
| `P<0.05` | `p` + `0.05` | a statistical claim becomes two tokens |
| `ALL` | `all` | **acute lymphoblastic leukemia becomes a stopword** |

That last one is the worst, because it is silent. A leukemia turns into the most common
determiner in English and nobody notices. The tokenizer guards against it with an explicit
list of medical abbreviations that survive lowercasing.

In [3]:
from src.tokenizer import PROTECTED_ABBREVIATIONS, smart_lower

print(f"{len(PROTECTED_ABBREVIATIONS)} protected abbreviations, e.g.:")
print("  ", sorted(PROTECTED_ABBREVIATIONS)[:18])

for token in ["ALL", "The", "HIV", "PATIENT", "TNF-alpha", "MS"]:
    print(f"  smart_lower({token!r}) -> {smart_lower(token)!r}")

112 protected abbreviations, e.g.:
   ['ACE', 'ADE', 'ADR', 'AE', 'AF', 'AIDS', 'AKI', 'ALL', 'ALP', 'ALS', 'ALT', 'AML', 'ARB', 'AST', 'AZT', 'BID', 'BP', 'BUN']
  smart_lower('ALL') -> 'ALL'
  smart_lower('The') -> 'the'
  smart_lower('HIV') -> 'HIV'
  smart_lower('PATIENT') -> 'patient'
  smart_lower('TNF-alpha') -> 'tnf-alpha'
  smart_lower('MS') -> 'MS'


The rule is narrow on purpose: all-caps, at most 5 characters, and **on the list**.
`PATIENT` is all-caps but not on the list, so it lowercases normally.

### What this costs, measured over the whole corpus

Keeping terms whole is not free. Measured over all 20,896 corpus sentences:

| Measure | Naive baseline | Domain tokenizer |
|---|---|---|
| Running tokens (words only) | 385,127 | **373,201** |
| Vocabulary (distinct types) | 17,198 | **20,320** |

Fewer tokens, because multi-part terms are not shattered into recycled fragments. But a
*larger* vocabulary, because `drug-induced` and `long-term` are now single types. A bigger
vocabulary means more embedding rows and more rare words — a real cost.

It is the right trade because **3,954 of those types cannot be represented at all** by the
naive pipeline. They are the domain-bearing ones, and they are exactly the words that
notebook 3's experiment is about.

### Offsets, not just words

`tokenize` returns each word's character position in the original string. This is not a
convenience — Stage 2 depends on it twice: to convert the corpus's character spans into
per-word tags, and to paint the predicted entities back onto the original sentence in the
demo.

In [4]:
text = "A case of recall pneumonitis induced by gemcitabine is reported."
tokens, offsets = tokenize(text)

for token, (start, end) in zip(tokens, offsets):
    print(f"  {token:14s} [{start:2d}:{end:2d}]  -> {text[start:end]!r}")

  a              [ 0: 1]  -> 'A'
  case           [ 2: 6]  -> 'case'
  of             [ 7: 9]  -> 'of'
  recall         [10:16]  -> 'recall'
  pneumonitis    [17:28]  -> 'pneumonitis'
  induced        [29:36]  -> 'induced'
  by             [37:39]  -> 'by'
  gemcitabine    [40:51]  -> 'gemcitabine'
  is             [52:54]  -> 'is'
  reported       [55:63]  -> 'reported'
  .              [63:64]  -> '.'


---

## 2.2 Sentence splitting

Splitting on `.` is wrong for medical abstracts. They are full of periods that are not
sentence boundaries: `i.v.`, `b.i.d.`, `Fig. 2`, `et al.`, `vs.`, and species names like
`E. coli`. Each bad split produces a fragment, and those fragments then pollute the
context windows the embeddings are trained on.

In [5]:
from src.tokenizer import sentence_split

abstract = ("The patient received 5 mg i.v. Two hours later she developed a rash. "
            "E. coli was isolated from blood cultures. See Fig. 2 for the time course. "
            "Smith et al. reported a similar case.")

for i, sentence in enumerate(sentence_split(abstract), 1):
    print(f"{i}. {sentence}")

print(f"\nnaive split on '. ' would give {len(abstract.split('. '))} pieces")

1. The patient received 5 mg i.v. Two hours later she developed a rash.
2. E. coli was isolated from blood cultures.
3. See Fig. 2 for the time course.
4. Smith et al. reported a similar case.

naive split on '. ' would give 8 pieces


Five real sentences, not eight fragments. Note `i.v. Two` — the hardest case, because the
next word *is* capitalised, so the usual "don't split before a lowercase word" heuristic
does not save you. It needs the explicit abbreviation guard.

### Preparing the PubMed corpus for embedding training

The same two functions are applied to all 159,975 abstracts to produce one sentence per
line, which is what `gensim` reads in notebook 3:

```python
for line in open("data/pubmed_corpus.jsonl"):
    record = json.loads(line)
    for sentence in sentence_split(record["title"] + " " + record["abstract"]):
        tokens = [t for t in tokenize(sentence)[0] if is_indexable(t)]
        if len(tokens) >= 3:                     # too short to be a useful context
            out.write(" ".join(tokens) + "\n")
```

| Measure | Value |
|---|---|
| Abstracts in | 159,975 |
| Sentences out | **1,883,683** |
| Running tokens | 40,519,109 |
| Mean sentence length | 21.5 tokens |
| Distinct words | 398,123 |
| Distinct words seen 5+ times | 113,103 |

Almost half the vocabulary (177,202 words, 44.5%) appears exactly once. For biomedical
text those are mostly author-specific compounds, OCR errors and one-off identifiers —
which is why Word2Vec's `min_count=5` discards them. FastText is the interesting
counter-case: it can still build a vector for them from character n-grams.

---

## 2.3 Character spans → BIO tags

This is the conversion that makes Stage 2 possible, and it is the single most delicate
piece of code in the project.

The corpus says *"characters 10 to 28 are an EFFECT"*. The tagger needs *"word 3 begins an
effect, word 4 continues it"*. That encoding is called **BIO** (or IOB2):

- `B-DRUG` — this word **b**egins a drug
- `I-DRUG` — this word is **i**nside the same drug
- `O` — **o**utside any entity

In [6]:
from src.bio import to_bio, TAGS

print("tag inventory:", TAGS)

text = "A case of recall pneumonitis induced by gemcitabine is reported."
spans = [(10, 28, "EFFECT"), (40, 51, "DRUG")]

tokens, tags = to_bio(text, spans)
print()
for token, tag in zip(tokens, tags):
    marker = "  <--" if tag != "O" else ""
    print(f"  {token:14s} {tag:9s}{marker}")

tag inventory: ('O', 'B-DRUG', 'I-DRUG', 'B-EFFECT', 'I-EFFECT')

  a              O        
  case           O        
  of             O        
  recall         B-EFFECT   <--
  pneumonitis    I-EFFECT   <--
  induced        O        
  by             O        
  gemcitabine    B-DRUG     <--
  is             O        
  reported       O        
  .              O        


### Two ways this goes silently wrong

**1. Containment instead of overlap.** The obvious way to decide whether a word belongs to
a span is "is the word entirely inside it". That drops any word straddling the boundary —
and the corpus does contain annotations that start or end mid-word. Nothing crashes; the
training labels are just quietly incomplete. `to_bio` tests for *overlap* instead.

**2. B- and I- assigned per label instead of per span.** Two adjacent DRUG entities must
produce `B-DRUG B-DRUG`. If the second one gets `I-DRUG`, the scorer reads them as one
entity and every count downstream is wrong.

In [7]:
# Two separate drugs, side by side.
text = "She received warfarin aspirin daily."
tokens, tags = to_bio(text, [(13, 21, "DRUG"), (22, 29, "DRUG")])
print(list(zip(tokens, tags)))

from src.bio import strict_entities
print("\nentities decoded back:", strict_entities(tags), "  <- two, not one")

[('she', 'O'), ('received', 'O'), ('warfarin', 'B-DRUG'), ('aspirin', 'B-DRUG'), ('daily', 'O'), ('.', 'O')]

entities decoded back: [(2, 3, 'DRUG'), (3, 4, 'DRUG')]   <- two, not one


### Nested entities: the one thing BIO genuinely cannot represent

The corpus annotates drug names *inside* effect phrases:

```
'theophylline intoxication'   EFFECT  [22:47]
'theophylline'                DRUG    [22:34]   <- nested inside the effect
```

One word can only carry one tag, so flat BIO cannot hold both. And if the inner span is
allowed to overwrite the middle of the outer one, the result is an `I-EFFECT` with no
`B-EFFECT` in front of it — a structurally impossible sequence, *in the gold labels*.

So overlaps are resolved before tagging: the longer span wins. The full adverse event is
kept, and the drug name is usually annotated elsewhere in the same sentence anyway. The
number of spans dropped this way is counted rather than assumed.

In [8]:
from src.bio import resolve_overlaps

nested = [(22, 47, "EFFECT"), (22, 34, "DRUG")]
kept, dropped = resolve_overlaps(nested)
print("kept   :", kept)
print("dropped:", dropped)

kept   : [(22, 47, 'EFFECT')]
dropped: [(22, 34, 'DRUG')]


### The conversion over the whole Stage 2 training split

The conversion is run over all 2,990 training sentences with a statistics accumulator, so
"nothing was lost" is a measured claim rather than an assumption.

In [9]:
from src.bio import ConversionStats

stats = ConversionStats()
train = pd.read_parquet(ROOT / "data" / "splits" / "stage2_train.parquet")

for _, row in train.iterrows():
    spans = [tuple(s) for s in json.loads(row["spans"])]
    to_bio(row["text"], spans, stats=stats, strict=False)

print(f"sentences converted     : {stats.sentences:,}")
print(f"spans seen              : {stats.spans:,}")
print(f"entities tagged         : {stats.entities_tagged:,}")
print(f"spans snapped to a word : {stats.spans_snapped:,}   (boundary fell mid-word)")
print(f"spans matching no word  : {stats.spans_unmatched:,}")
print(f"nested spans dropped    : {stats.spans_dropped_nested:,}  {dict(stats.dropped_by_label)}")

print("\nexamples of nested spans that were dropped:")
for line in stats.examples_dropped[:3]:
    print("  -", line[:110])

sentences converted     : 2,990
spans seen              : 7,739
entities tagged         : 7,620
spans snapped to a word : 511   (boundary fell mid-word)
spans matching no word  : 0
nested spans dropped    : 119  {'DRUG': 106, 'EFFECT': 13}

examples of nested spans that were dropped:
  - DRUG 'theophylline' nested in 'A case is reported of theophylline intoxication due to a dra'
  - DRUG 'theophylline' nested in 'A case is reported of theophylline intoxication due to a dra'
  - DRUG 'theophylline' nested in 'A diagnosis of masked theophylline poisoning should be consi'


---

## What this notebook produced

| Piece | Where it lives | Used by |
|---|---|---|
| domain tokenizer with offsets | `src/tokenizer.py` | everything |
| sentence splitter | `src/tokenizer.py` | corpus prep, the demo |
| `data/pubmed_sentences.txt` | 1.88M sentences, one per line | notebook 3 |
| character spans → BIO tags | `src/bio.py` | notebook 5 |

**Next:** [3 — Embeddings](03_embeddings.ipynb), the experiment the whole project is built
around.